# Tutorial 9: Object Detection using SSD & YOLO (Custom Model)

**Objectives**
- Understand the backbones of SSD & YOLO
- Understand detection heads and prediction layers
- Implement SSD & YOLO from scratch

**Tasks (what this notebook does)**
1. Take a **Kaggle dataset** (Face Mask Detection, 853 images, 3 classes, Pascal VOC boxes) and **test it on the models built in the tutorial**, then **improve them by adding / removing layers**.
2. Take **one pretrained YOLO (YOLOv8n)** and **one pretrained SSD (SSDLite320 MobileNetV3)**, fine-tune each on the same data and test.
3. Compare all models with the same metric (**mAP@0.5**) and the same train/validation split.

**Notebook map**
| Part | Content |
|---|---|
| 0 | Setup and configuration |
| 1 | Kaggle dataset: download, parse, visualize |
| 2 | Shared utilities (augmentation, mAP, training loop) |
| 3 | YOLO from scratch: tutorial version → improved version |
| 4 | SSD from scratch: tutorial version → improved version |
| 5 | Pretrained YOLOv8n fine-tuned |
| 6 | Pretrained SSDLite fine-tuned |
| 7 | Comparison and visual results |

> Use **Runtime → Change runtime type → GPU (T4)** before running. Total run time is roughly 30-50 minutes on a T4.

## Part 0: Setup

In [ ]:
!pip install -q ultralytics kagglehub

In [ ]:
import os, glob, time, random, math, shutil, subprocess
import xml.etree.ElementTree as ET
from functools import partial
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
AUTOTUNE = tf.data.AUTOTUNE

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

In [ ]:
# ---------------- Configuration ----------------
CLASS_NAMES = ['with_mask', 'without_mask', 'mask_weared_incorrect']
NUM_CLASSES = len(CLASS_NAMES)

YOLO_SIZE, GRID, YOLO_BOXES = 416, 13, 3     # 416 / 32 = 13x13 grid (as in the tutorial)
SSD_SIZE, SSD_BOXES = 256, 6                 # 6 anchors per cell (as in the tutorial)
BATCH = 16

EPOCHS_SCRATCH = 40        # models built from scratch
EPOCHS_PRETRAINED = 25     # fine-tuning pretrained YOLO / SSD

# Store every model's results here for the final comparison
RESULTS, ALL_PREDS, HISTORIES = {}, {}, {}

## Part 1: Kaggle dataset - Face Mask Detection

**Source:** [andrewmvd/face-mask-detection](https://www.kaggle.com/datasets/andrewmvd/face-mask-detection) - 853 images, Pascal VOC XML boxes, 3 classes:
`with_mask`, `without_mask`, `mask_weared_incorrect`.

The cell below tries `kagglehub` first. If Kaggle asks for credentials, it falls back to the Kaggle API: create a token at *Kaggle → Settings → API → Create New Token* and upload the `kaggle.json` file when prompted.

In [ ]:
DATA_ROOT = None
try:
    import kagglehub
    DATA_ROOT = kagglehub.dataset_download("andrewmvd/face-mask-detection")
except Exception as e:
    print("kagglehub failed (", e, ") -> using the Kaggle API instead")
    from google.colab import files
    print("Upload your kaggle.json now")
    files.upload()
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    shutil.move('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
    subprocess.run(['kaggle', 'datasets', 'download', '-d', 'andrewmvd/face-mask-detection',
                    '-p', '/content/fmd', '--unzip'], check=True)
    DATA_ROOT = '/content/fmd'
print("Dataset folder:", DATA_ROOT)
print(os.listdir(DATA_ROOT))

In [ ]:
# Parse Pascal VOC XML -> list of items with NORMALIZED boxes [x1, y1, x2, y2] in [0, 1]
def parse_voc(xml_path):
    root = ET.parse(xml_path).getroot()
    W = float(root.find('size').find('width').text)
    H = float(root.find('size').find('height').text)
    boxes, labels = [], []
    for obj in root.findall('object'):
        name = obj.find('name').text.strip()
        if name not in CLASS_NAMES:
            continue
        bb = obj.find('bndbox')
        x1, y1, x2, y2 = [float(bb.find(k).text) for k in ('xmin', 'ymin', 'xmax', 'ymax')]
        x1, x2 = np.clip([x1 / W, x2 / W], 0, 1)
        y1, y2 = np.clip([y1 / H, y2 / H], 0, 1)
        if x2 - x1 < 0.002 or y2 - y1 < 0.002:      # skip degenerate boxes
            continue
        boxes.append([x1, y1, x2, y2]); labels.append(CLASS_NAMES.index(name))
    return root.find('filename').text.strip(), boxes, labels

img_lookup = {os.path.basename(p): p
              for p in glob.glob(os.path.join(DATA_ROOT, '**', '*'), recursive=True)
              if p.lower().endswith(('.png', '.jpg', '.jpeg'))}
xml_paths = sorted(glob.glob(os.path.join(DATA_ROOT, '**', '*.xml'), recursive=True))

all_items = []
for xp in xml_paths:
    fname, boxes, labels = parse_voc(xp)
    if fname in img_lookup and len(boxes) > 0:
        all_items.append(dict(path=img_lookup[fname], boxes=boxes, labels=labels))
print("Usable images:", len(all_items), "of", len(xml_paths), "annotation files")

# Fixed train / validation split (80 / 20) used by EVERY model
rng = np.random.RandomState(SEED)
perm = rng.permutation(len(all_items))
n_train = int(0.8 * len(all_items))
train_items = [all_items[i] for i in perm[:n_train]]
val_items   = [all_items[i] for i in perm[n_train:]]
print("Train:", len(train_items), " Val:", len(val_items))

In [ ]:
# Dataset statistics
counts = np.zeros(NUM_CLASSES, int)
sizes, per_img = [], []
for it in all_items:
    for b, l in zip(it['boxes'], it['labels']):
        counts[l] += 1; sizes.append([b[2] - b[0], b[3] - b[1]])
    per_img.append(len(it['labels']))
sizes = np.array(sizes)
print(dict(zip(CLASS_NAMES, counts)))
print(f"Boxes per image: mean={np.mean(per_img):.1f}, max={max(per_img)}")
print(f"Median box size (fraction of image): w={np.median(sizes[:,0]):.3f}, h={np.median(sizes[:,1]):.3f}")

COLORS = [(0, 200, 0), (230, 0, 0), (255, 190, 0)]   # with / without / incorrect

def draw_boxes(img, boxes, labels, scores=None, thr=0.0):
    img = img.copy(); H, W = img.shape[:2]
    for i, (b, l) in enumerate(zip(boxes, labels)):
        if scores is not None and scores[i] < thr:
            continue
        x1, y1, x2, y2 = (np.array(b) * [W, H, W, H]).astype(int)
        cv2.rectangle(img, (x1, y1), (x2, y2), COLORS[int(l) % 3], max(1, W // 150))
        if scores is not None:
            cv2.putText(img, f"{scores[i]:.2f}", (x1, max(10, y1 - 3)), cv2.FONT_HERSHEY_SIMPLEX,
                        max(0.35, W / 900), COLORS[int(l) % 3], 1)
    return img

fig, ax = plt.subplots(2, 4, figsize=(16, 7))
for a, it in zip(ax.ravel(), train_items[:8]):
    im = cv2.cvtColor(cv2.imread(it['path']), cv2.COLOR_BGR2RGB)
    a.imshow(draw_boxes(im, it['boxes'], it['labels'])); a.axis('off')
fig.suptitle("Green = with_mask, Red = without_mask, Yellow = mask_weared_incorrect")
plt.tight_layout(); plt.show()

## Part 2: Shared utilities

- images are resized to a square input and normalized to [0, 1]
- ground truth is padded to a fixed number of boxes per image (label `0` = padding, classes become `1..C`)
- **augmentation** (training only): random horizontal flip + brightness jitter
- **mAP@0.5** (Pascal VOC style, all-point interpolation) is implemented once and used for every model, so the comparison is fair
- a small custom training loop is used so the same code can train SSD (two outputs) and YOLO (one output)

In [ ]:
def load_images(items, size):
    X = np.zeros((len(items), size, size, 3), np.uint8)
    for i, it in enumerate(items):
        im = cv2.cvtColor(cv2.imread(it['path']), cv2.COLOR_BGR2RGB)
        X[i] = cv2.resize(im, (size, size), interpolation=cv2.INTER_AREA)
    return X

MAXM = max(len(it['labels']) for it in all_items)

def pad_gt(items, maxm=MAXM):
    B = np.zeros((len(items), maxm, 4), np.float32)
    L = np.zeros((len(items), maxm), np.int32)          # 0 = padding, classes are 1..C
    for i, it in enumerate(items):
        m = len(it['labels'])
        B[i, :m] = it['boxes']; L[i, :m] = np.array(it['labels']) + 1
    return B, L

def augment(x, b, l):
    x = tf.cast(x, tf.float32) / 255.0
    flip = tf.random.uniform([]) < 0.5
    x = tf.cond(flip, lambda: x[:, ::-1, :], lambda: x)
    b = tf.cond(flip, lambda: tf.stack([1 - b[:, 2], b[:, 1], 1 - b[:, 0], b[:, 3]], axis=1), lambda: b)
    x = tf.clip_by_value(tf.image.random_brightness(x, 0.15), 0.0, 1.0)
    return x, b, l

def build_ds(X, B, L, encoder, train, batch=BATCH):
    ds = tf.data.Dataset.from_tensor_slices((X, B, L))
    if train:
        ds = ds.shuffle(len(X), seed=SEED, reshuffle_each_iteration=True).map(augment, AUTOTUNE)
    else:
        ds = ds.map(lambda x, b, l: (tf.cast(x, tf.float32) / 255.0, b, l), AUTOTUNE)
    ds = ds.map(lambda x, b, l: (x, encoder(b, l)), AUTOTUNE)
    return ds.batch(batch).prefetch(AUTOTUNE)

# ---------------- mAP@0.5 ----------------
def _iou_one_to_many(b, boxes):
    x1 = np.maximum(b[0], boxes[:, 0]); y1 = np.maximum(b[1], boxes[:, 1])
    x2 = np.minimum(b[2], boxes[:, 2]); y2 = np.minimum(b[3], boxes[:, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    a = (b[2] - b[0]) * (b[3] - b[1])
    g = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    return inter / (a + g - inter + 1e-9)

def compute_map(preds, items, num_classes=NUM_CLASSES, iou_thr=0.5):
    # preds: list (one per image) of (boxes_xyxy_normalized, scores, class_ids 0-based)
    gts = [(np.array(it['boxes'], np.float32), np.array(it['labels'])) for it in items]
    aps = []
    for c in range(num_classes):
        gt_c = {i: g[0][g[1] == c] for i, g in enumerate(gts)}
        n_gt = sum(len(v) for v in gt_c.values())
        dets = [(s, i, b) for i, (bx, sc, cl) in enumerate(preds)
                for b, s, k in zip(bx, sc, cl) if int(k) == c]
        dets.sort(key=lambda t: -t[0])
        used = {i: np.zeros(len(v), bool) for i, v in gt_c.items()}
        tp = np.zeros(len(dets)); fp = np.zeros(len(dets))
        for d, (s, i, b) in enumerate(dets):
            g = gt_c[i]
            if len(g) == 0:
                fp[d] = 1; continue
            ious = _iou_one_to_many(b, g); j = int(ious.argmax())
            if ious[j] >= iou_thr and not used[i][j]:
                tp[d] = 1; used[i][j] = True
            else:
                fp[d] = 1
        if n_gt == 0:
            aps.append(float('nan')); continue
        tp, fp = np.cumsum(tp), np.cumsum(fp)
        rec = tp / n_gt; prec = tp / np.maximum(tp + fp, 1e-9)
        mrec = np.concatenate([[0.0], rec, [1.0]]); mpre = np.concatenate([[0.0], prec, [0.0]])
        for k in range(len(mpre) - 2, -1, -1):
            mpre[k] = max(mpre[k], mpre[k + 1])
        idx = np.where(mrec[1:] != mrec[:-1])[0]
        aps.append(float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1])))
    return float(np.nanmean(aps)), aps

# ---------------- Training loop (works for single- or multi-output models) ----------------
def train_model(model, loss_fn, train_ds, val_ds, epochs, lr, name):
    steps = max(1, sum(1 for _ in train_ds)) * epochs
    schedule = tf.keras.optimizers.schedules.CosineDecay(lr, decay_steps=steps, alpha=0.05)
    opt = tf.keras.optimizers.Adam(learning_rate=schedule, clipnorm=10.0)

    @tf.function
    def train_step(x, y):
        with tf.GradientTape() as tape:
            loss = loss_fn(y, model(x, training=True))
        grads = tape.gradient(loss, model.trainable_variables)
        opt.apply_gradients(zip(grads, model.trainable_variables))
        return loss

    @tf.function
    def val_step(x, y):
        return loss_fn(y, model(x, training=False))

    hist = {'loss': [], 'val_loss': []}
    best, best_w = float('inf'), None
    t0 = time.time()
    for ep in range(epochs):
        tl = np.mean([float(train_step(x, y)) for x, y in train_ds])
        vl = np.mean([float(val_step(x, y)) for x, y in val_ds])
        hist['loss'].append(tl); hist['val_loss'].append(vl)
        if vl < best:
            best, best_w = vl, model.get_weights()
        if ep % 5 == 0 or ep == epochs - 1:
            print(f"[{name}] epoch {ep+1:3d}/{epochs}  loss={tl:.4f}  val_loss={vl:.4f}")
    model.set_weights(best_w)       # keep the best validation checkpoint
    return hist, time.time() - t0

def predict_detections(model, X, decode_fn, conf=0.05, batch=16):
    out = []
    for i in range(0, len(X), batch):
        xb = X[i:i + batch].astype(np.float32) / 255.0
        boxes, scores, classes, valid = decode_fn(model(xb, training=False), conf)
        for j in range(len(xb)):
            n = int(valid[j])
            out.append((boxes[j, :n].numpy(), scores[j, :n].numpy(), classes[j, :n].numpy().astype(int)))
    return out

def register(name, preds, params, train_time, hist=None):
    m, aps = compute_map(preds, val_items)
    RESULTS[name] = dict(params=params, train_time_s=round(train_time), **{'mAP@0.5': m},
                         **{f'AP {c}': a for c, a in zip(CLASS_NAMES, aps)})
    ALL_PREDS[name] = preds
    if hist is not None:
        HISTORIES[name] = hist
    print(f"==> {name}: mAP@0.5 = {m:.4f}   per class: " + ", ".join(f"{a:.3f}" for a in aps))

def plot_history(names):
    fig, ax = plt.subplots(1, len(names), figsize=(6 * len(names), 3.5))
    ax = np.atleast_1d(ax)
    for a, n in zip(ax, names):
        a.plot(HISTORIES[n]['loss'], label='train'); a.plot(HISTORIES[n]['val_loss'], label='val')
        a.set_title(n); a.set_xlabel('epoch'); a.set_yscale('log'); a.legend()
    plt.tight_layout(); plt.show()

In [ ]:
# Prepare arrays once (images at both input sizes + padded ground truth)
Btr, Ltr = pad_gt(train_items); Bva, Lva = pad_gt(val_items)
X_tr_yolo, X_va_yolo = load_images(train_items, YOLO_SIZE), load_images(val_items, YOLO_SIZE)
X_tr_ssd,  X_va_ssd  = load_images(train_items, SSD_SIZE),  load_images(val_items, SSD_SIZE)
print("YOLO inputs:", X_tr_yolo.shape, X_va_yolo.shape)
print("SSD  inputs:", X_tr_ssd.shape, X_va_ssd.shape)

## Part 3: YOLO from scratch

**Idea:** the backbone shrinks a 416×416 image to a 13×13 feature map (5 max-pools: 416/32 = 13). Each of the 13×13 grid cells predicts `3` boxes. Each box has `4` coordinates + `1` objectness + `C` class scores, so the head output is `(13, 13, 3, 5 + C)`.

### 3.1 Tutorial architecture (baseline) - exactly as in the PDF

In [ ]:
# Step 1: Define a simple backbone for YOLO (from the tutorial)
def build_yolo_backbone(input_shape=(416, 416, 3)):
    inputs = tf.keras.Input(shape=input_shape)
    x = layers.Conv2D(32, (3, 3), strides=1, padding="same", activation="relu")(inputs)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), strides=1, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(128, (3, 3), strides=1, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(256, (3, 3), strides=1, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(512, (3, 3), strides=1, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    return models.Model(inputs, x, name="YOLO_Backbone")

# Step 2: Define YOLO head (detection layer)
def yolo_head(x, num_classes, num_boxes):
    num_outputs = num_boxes * (num_classes + 5)      # 5 = (x, y, w, h) + confidence
    yolo_output = layers.Conv2D(num_outputs, (1, 1), padding="same")(x)
    yolo_output = layers.Reshape((GRID, GRID, num_boxes, num_classes + 5))(yolo_output)
    return yolo_output

# Step 3: Combine backbone and YOLO head
def build_yolo_model(input_shape=(416, 416, 3), num_classes=NUM_CLASSES, num_boxes=YOLO_BOXES):
    backbone = build_yolo_backbone(input_shape)
    output = yolo_head(backbone.output, num_classes, num_boxes)
    return models.Model(inputs=backbone.input, outputs=output, name="YOLO_Baseline")

yolo_base = build_yolo_model()
yolo_base.summary()

# Step 4: dummy data -> verify output shape
dummy_input = np.random.random((1, 416, 416, 3))
print("YOLO Prediction Shape:", yolo_base.predict(dummy_input, verbose=0).shape)   # (1, 13, 13, 3, 5 + C)

### 3.2 What the tutorial does not include: anchors, targets, loss, decoding

To *train* the model on real data we need four extra pieces.

1. **Anchors:** 3 reference box shapes, found by **k-means on the training box sizes** (so they fit faces).
2. **Target encoding:** each ground-truth box is assigned to the grid cell that contains its centre and to the anchor with the best shape match. Targets: `tx, ty` (offset inside the cell), `tw, th = log(w / anchor_w)`, objectness 1, class one-hot.
3. **Loss:** squared error for box coordinates, binary cross-entropy for objectness (cells without objects weighted 0.5) and for classes (only for cells with objects).
4. **Decoding + NMS:** convert raw outputs back to boxes and remove duplicates with non-max suppression.

In [ ]:
from sklearn.cluster import KMeans

wh_train = np.array([[b[2] - b[0], b[3] - b[1]] for it in train_items for b in it['boxes']])
km = KMeans(n_clusters=YOLO_BOXES, n_init=10, random_state=SEED).fit(wh_train)
YOLO_ANCHORS = km.cluster_centers_[np.argsort(km.cluster_centers_.prod(axis=1))].astype(np.float32)
print("YOLO anchors (w, h as fraction of image):\n", YOLO_ANCHORS.round(3))

def make_yolo_encoder(anchors, grid=GRID, num_classes=NUM_CLASSES):
    anchors = tf.constant(anchors, tf.float32)
    nb = anchors.shape[0]
    def encode(gt_boxes, gt_labels):                     # (M, 4), (M,)
        valid = gt_labels > 0
        boxes = tf.boolean_mask(gt_boxes, valid)
        labels = tf.boolean_mask(gt_labels, valid) - 1
        cx = (boxes[:, 0] + boxes[:, 2]) / 2; cy = (boxes[:, 1] + boxes[:, 3]) / 2
        w = tf.maximum(boxes[:, 2] - boxes[:, 0], 1e-4); h = tf.maximum(boxes[:, 3] - boxes[:, 1], 1e-4)
        gx, gy = cx * grid, cy * grid
        col = tf.clip_by_value(tf.cast(tf.floor(gx), tf.int32), 0, grid - 1)
        row = tf.clip_by_value(tf.cast(tf.floor(gy), tf.int32), 0, grid - 1)
        # best anchor = highest IoU of widths/heights only
        inter = tf.minimum(w[:, None], anchors[None, :, 0]) * tf.minimum(h[:, None], anchors[None, :, 1])
        union = w[:, None] * h[:, None] + anchors[None, :, 0] * anchors[None, :, 1] - inter
        best = tf.argmax(inter / union, axis=1, output_type=tf.int32)
        tx = gx - tf.cast(col, tf.float32); ty = gy - tf.cast(row, tf.float32)
        tw = tf.math.log(w / tf.gather(anchors[:, 0], best))
        th = tf.math.log(h / tf.gather(anchors[:, 1], best))
        upd = tf.concat([tf.stack([tx, ty, tw, th, tf.ones_like(tx)], axis=1),
                         tf.one_hot(labels, num_classes)], axis=1)
        idx = tf.stack([row, col, best], axis=1)
        y = tf.zeros([grid, grid, nb, 5 + num_classes])
        return tf.tensor_scatter_nd_update(y, idx, upd)
    return encode

def yolo_loss(y_true, y_pred, l_coord=5.0, l_noobj=0.5):
    obj = y_true[..., 4:5]
    xy_loss = tf.reduce_sum(obj * tf.square(y_true[..., 0:2] - tf.sigmoid(y_pred[..., 0:2])), axis=[1, 2, 3, 4])
    wh_loss = tf.reduce_sum(obj * tf.square(y_true[..., 2:4] - y_pred[..., 2:4]), axis=[1, 2, 3, 4])
    obj_ce = tf.nn.sigmoid_cross_entropy_with_logits(labels=obj, logits=y_pred[..., 4:5])
    obj_loss = tf.reduce_sum((obj + l_noobj * (1 - obj)) * obj_ce, axis=[1, 2, 3, 4])
    cls_ce = tf.nn.sigmoid_cross_entropy_with_logits(labels=y_true[..., 5:], logits=y_pred[..., 5:])
    cls_loss = tf.reduce_sum(obj * cls_ce, axis=[1, 2, 3, 4])
    return tf.reduce_mean(l_coord * (xy_loss + wh_loss) + obj_loss + cls_loss)

def decode_yolo(raw, conf_thr=0.05, anchors=YOLO_ANCHORS):
    anchors = tf.constant(anchors, tf.float32)
    g = raw.shape[1]; C = raw.shape[-1] - 5
    xy = tf.sigmoid(raw[..., 0:2])
    wh = tf.exp(tf.clip_by_value(raw[..., 2:4], -8.0, 4.0)) * anchors
    gx, gy = tf.meshgrid(tf.range(g, dtype=tf.float32), tf.range(g, dtype=tf.float32))
    cx = (gx[None, :, :, None] + xy[..., 0]) / g
    cy = (gy[None, :, :, None] + xy[..., 1]) / g
    w, h = wh[..., 0], wh[..., 1]
    boxes = tf.stack([cy - h / 2, cx - w / 2, cy + h / 2, cx + w / 2], axis=-1)      # y1 x1 y2 x2
    scores = tf.sigmoid(raw[..., 4:5]) * tf.sigmoid(raw[..., 5:])
    B = tf.shape(raw)[0]
    nms = tf.image.combined_non_max_suppression(
        tf.reshape(boxes, [B, -1, 1, 4]), tf.reshape(scores, [B, -1, C]),
        max_output_size_per_class=50, max_total_size=100,
        iou_threshold=0.45, score_threshold=conf_thr, clip_boxes=True)
    b = nms.nmsed_boxes
    xyxy = tf.stack([b[..., 1], b[..., 0], b[..., 3], b[..., 2]], axis=-1)
    return xyxy, nms.nmsed_scores, nms.nmsed_classes, nms.valid_detections

yolo_encoder = make_yolo_encoder(YOLO_ANCHORS)
yolo_train_ds = build_ds(X_tr_yolo, Btr, Ltr, yolo_encoder, train=True)
yolo_val_ds   = build_ds(X_va_yolo, Bva, Lva, yolo_encoder, train=False)
print("Target shape per image:", next(iter(yolo_train_ds))[1].shape[1:])

### 3.3 Improved YOLO: adding / removing layers

Changes compared with the tutorial backbone (each is a layer-level change you can discuss in your report):

| Change | Why |
|---|---|
| **+ BatchNorm** after every conv (conv without bias) | stabilises and speeds up training from scratch |
| **ReLU → LeakyReLU(0.1)** | avoids "dead" neurons (as in Darknet) |
| **+ 1×1 bottleneck convs** in the 256 and 512 stages (3×3 → 1×1 → 3×3) | more depth with few extra parameters |
| **+ extra 3×3 conv before the head** | gives the head more capacity |
| **− bias in conv layers** | BatchNorm already provides it |

In [ ]:
def conv_bn(x, f, k=3, s=1):
    x = layers.Conv2D(f, k, strides=s, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return layers.LeakyReLU(0.1)(x)

def build_yolo_improved(input_shape=(416, 416, 3), num_classes=NUM_CLASSES, num_boxes=YOLO_BOXES):
    inputs = tf.keras.Input(shape=input_shape)
    x = inputs
    for f in (32, 64, 128):
        x = conv_bn(x, f); x = layers.MaxPooling2D(2)(x)
    for f in (256, 512):
        x = conv_bn(x, f); x = conv_bn(x, f // 2, 1); x = conv_bn(x, f)     # 3x3 -> 1x1 -> 3x3
        x = layers.MaxPooling2D(2)(x)
    x = conv_bn(x, 512)                                                      # extra conv before the head
    out = yolo_head(x, num_classes, num_boxes)
    return models.Model(inputs, out, name="YOLO_Improved")

yolo_imp = build_yolo_improved()
print("Baseline params:", f"{yolo_base.count_params():,}", " Improved params:", f"{yolo_imp.count_params():,}")
print("Improved output shape:", yolo_imp.output_shape)

In [ ]:
# Train both YOLO versions from scratch
yolo_cfg = {'YOLO baseline (tutorial)': (yolo_base, 3e-4), 'YOLO improved': (yolo_imp, 1e-3)}
for name, (m, lr) in yolo_cfg.items():
    tf.random.set_seed(SEED)
    hist, t = train_model(m, yolo_loss, yolo_train_ds, yolo_val_ds, EPOCHS_SCRATCH, lr, name)
    preds = predict_detections(m, X_va_yolo, decode_yolo)
    register(name, preds, m.count_params(), t, hist)
plot_history(list(yolo_cfg))

## Part 4: SSD from scratch

**Idea:** SSD predicts boxes from **several feature maps of different sizes** - large maps find small objects, small maps find large objects. Every cell of every map has `6` anchor boxes (different sizes and aspect ratios). For each anchor two heads predict: **class scores** (`C + 1`, the extra one is *background*) and **4 box offsets**.

### 4.1 Tutorial architecture (baseline)

The tutorial's backbone is kept as is. Input is 256×256 (instead of 300) so the feature-map sizes are whole numbers: 64 → 32 → 16.

In [ ]:
# Step 1: Define a simple backbone network for SSD (from the tutorial)
def build_backbone(input_shape=(256, 256, 3)):
    inputs = tf.keras.Input(shape=input_shape)
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    return models.Model(inputs, x, name="SSD_Backbone")

# Step 2: Define SSD prediction head for class and bounding box predictions
def ssd_head(x, num_classes, num_boxes):
    class_head = layers.Conv2D(num_boxes * num_classes, (3, 3), padding="same")(x)
    class_head = layers.Reshape((-1, num_classes))(class_head)
    box_head = layers.Conv2D(num_boxes * 4, (3, 3), padding="same")(x)      # 4 values per anchor
    box_head = layers.Reshape((-1, 4))(box_head)
    return class_head, box_head

# Step 3: Build the SSD model (backbone + heads on multi-scale feature maps)
def build_ssd_model(input_shape=(SSD_SIZE, SSD_SIZE, 3), num_classes=NUM_CLASSES + 1, num_boxes=SSD_BOXES):
    base = build_backbone(input_shape)
    x = base.output
    class_outputs, box_outputs, fmap_sizes = [], [], []
    for i in range(3):                                   # 3 scales
        fmap_sizes.append(x.shape[1])
        c, b = ssd_head(x, num_classes, num_boxes)
        class_outputs.append(c); box_outputs.append(b)
        if i < 2:                                        # downsample for the next scale
            x = layers.Conv2D(128, (3, 3), strides=2, padding="same", activation='relu')(x)
    classes_concat = layers.Concatenate(axis=1)(class_outputs)
    boxes_concat = layers.Concatenate(axis=1)(box_outputs)
    return models.Model(inputs=base.input, outputs=[classes_concat, boxes_concat], name="SSD_Baseline"), fmap_sizes

ssd_base, base_fmaps = build_ssd_model()
ssd_base.summary()

# Step 4: dummy data -> verify output shapes
ssd_class_pred, ssd_box_pred = ssd_base.predict(np.random.random((1, SSD_SIZE, SSD_SIZE, 3)), verbose=0)
print("\nSSD Class Prediction Shape:", ssd_class_pred.shape)   # (1, num_anchors, C+1)
print("SSD Box Prediction Shape:", ssd_box_pred.shape)         # (1, num_anchors, 4)
print("Feature map sizes:", base_fmaps)

### 4.2 Anchors, matching, loss and decoding

- **Anchors:** for every cell: two square boxes (size `s` and `sqrt(s·s')`) plus aspect ratios 2, 3, 1/2, 1/3 → 6 anchors.
- **Matching:** every ground-truth box is matched to its best anchor, plus every anchor with IoU ≥ 0.5. Unmatched anchors are *background*.
- **Loss:** Smooth-L1 for box offsets (positives only) + softmax cross-entropy for classes with **hard negative mining** (only the 3× hardest background anchors are used, because background anchors vastly outnumber positives).
- **Decoding:** softmax probabilities + offsets → boxes → non-max suppression.

In [ ]:
def make_anchors(fmap_sizes, scales, ratios=(2, 3, 0.5, 1 / 3)):
    # returns (N, 4) anchors as (cx, cy, w, h), ordered (y, x, anchor) to match the Reshape of the heads
    out = []
    for k, f in enumerate(fmap_sizes):
        s, s2 = scales[k], math.sqrt(scales[k] * scales[k + 1])
        wh = [(s, s), (s2, s2)] + [(s * math.sqrt(r), s / math.sqrt(r)) for r in ratios]
        for y in range(f):
            for x in range(f):
                for (w, h) in wh:
                    out.append([(x + 0.5) / f, (y + 0.5) / f, w, h])
    return np.array(out, np.float32)

VAR = (0.1, 0.2)   # SSD box-encoding variances

def make_ssd_encoder(anchors, iou_thr=0.5):
    A = tf.constant(anchors)
    a_xyxy = tf.concat([A[:, :2] - A[:, 2:] / 2, A[:, :2] + A[:, 2:] / 2], axis=1)
    a_area = A[:, 2] * A[:, 3]
    def encode(gt_boxes, gt_labels):                     # (M, 4), (M,) with 0 = padding
        valid = gt_labels > 0
        x1 = tf.maximum(a_xyxy[:, None, 0], gt_boxes[None, :, 0]); y1 = tf.maximum(a_xyxy[:, None, 1], gt_boxes[None, :, 1])
        x2 = tf.minimum(a_xyxy[:, None, 2], gt_boxes[None, :, 2]); y2 = tf.minimum(a_xyxy[:, None, 3], gt_boxes[None, :, 3])
        inter = tf.maximum(x2 - x1, 0) * tf.maximum(y2 - y1, 0)
        g_area = (gt_boxes[:, 2] - gt_boxes[:, 0]) * (gt_boxes[:, 3] - gt_boxes[:, 1])
        iou = inter / (a_area[:, None] + g_area[None, :] - inter + 1e-9)          # (A, M)
        iou = tf.where(valid[None, :], iou, -1.0)
        best_gt = tf.argmax(iou, axis=1, output_type=tf.int32)
        best_iou = tf.reduce_max(iou, axis=1)
        # force every ground-truth box to keep its best anchor
        best_anchor = tf.argmax(iou, axis=0, output_type=tf.int32)
        vidx = tf.cast(tf.where(valid)[:, 0], tf.int32)
        anc = tf.gather(best_anchor, vidx)
        best_iou = tf.tensor_scatter_nd_update(best_iou, anc[:, None], tf.fill(tf.shape(anc), 2.0))
        best_gt = tf.tensor_scatter_nd_update(best_gt, anc[:, None], vidx)
        pos = best_iou >= iou_thr
        mb = tf.gather(gt_boxes, best_gt); ml = tf.gather(gt_labels, best_gt)
        cls_t = tf.where(pos, ml, tf.zeros_like(ml))                               # 0 = background
        gcx = (mb[:, 0] + mb[:, 2]) / 2; gcy = (mb[:, 1] + mb[:, 3]) / 2
        gw = tf.maximum(mb[:, 2] - mb[:, 0], 1e-4); gh = tf.maximum(mb[:, 3] - mb[:, 1], 1e-4)
        box_t = tf.stack([(gcx - A[:, 0]) / (A[:, 2] * VAR[0]), (gcy - A[:, 1]) / (A[:, 3] * VAR[0]),
                          tf.math.log(gw / A[:, 2]) / VAR[1], tf.math.log(gh / A[:, 3]) / VAR[1]], axis=1)
        return cls_t, box_t
    return encode

def ssd_loss(y_true, y_pred, neg_ratio=3):
    cls_t, box_t = y_true
    cls_p, box_p = y_pred
    pos = tf.cast(cls_t > 0, tf.float32)                                           # (B, A)
    num_pos = tf.maximum(tf.reduce_sum(pos, axis=1), 1.0)
    ce = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=cls_t, logits=cls_p)
    # hard negative mining: keep the neg_ratio*num_pos background anchors with the largest loss
    neg_ce = ce * (1.0 - pos)
    rank = tf.argsort(tf.argsort(neg_ce, axis=1, direction='DESCENDING'), axis=1)
    neg_mask = tf.cast(rank < tf.cast(neg_ratio * num_pos[:, None], tf.int32), tf.float32) * (1.0 - pos)
    cls_loss = tf.reduce_sum(ce * (pos + neg_mask), axis=1) / num_pos
    d = box_p - box_t; ad = tf.abs(d)
    smooth_l1 = tf.where(ad < 1.0, 0.5 * d * d, ad - 0.5)
    loc_loss = tf.reduce_sum(tf.reduce_sum(smooth_l1, axis=-1) * pos, axis=1) / num_pos
    return tf.reduce_mean(cls_loss + loc_loss)

def decode_ssd(raw, conf_thr, anchors):
    cls_p, box_p = raw
    A = tf.constant(anchors)
    probs = tf.nn.softmax(cls_p, axis=-1)[..., 1:]                                 # drop background
    cx = box_p[..., 0] * VAR[0] * A[:, 2] + A[:, 0]
    cy = box_p[..., 1] * VAR[0] * A[:, 3] + A[:, 1]
    w = tf.exp(tf.clip_by_value(box_p[..., 2] * VAR[1], -8.0, 4.0)) * A[:, 2]
    h = tf.exp(tf.clip_by_value(box_p[..., 3] * VAR[1], -8.0, 4.0)) * A[:, 3]
    boxes = tf.stack([cy - h / 2, cx - w / 2, cy + h / 2, cx + w / 2], axis=-1)[:, :, None, :]
    nms = tf.image.combined_non_max_suppression(
        boxes, probs, max_output_size_per_class=50, max_total_size=100,
        iou_threshold=0.45, score_threshold=conf_thr, clip_boxes=True)
    b = nms.nmsed_boxes
    xyxy = tf.stack([b[..., 1], b[..., 0], b[..., 3], b[..., 2]], axis=-1)
    return xyxy, nms.nmsed_scores, nms.nmsed_classes, nms.valid_detections

# anchors + datasets for the baseline (3 scales: 64, 32, 16)
SSD_BASE_SCALES = [0.06, 0.15, 0.35, 0.70]
anchors_base = make_anchors(base_fmaps, SSD_BASE_SCALES)
print("Baseline anchors:", anchors_base.shape, " model outputs:", ssd_class_pred.shape[1])
assert anchors_base.shape[0] == ssd_class_pred.shape[1]

enc_base = make_ssd_encoder(anchors_base)
ssd_train_ds_base = build_ds(X_tr_ssd, Btr, Ltr, enc_base, train=True)
ssd_val_ds_base   = build_ds(X_va_ssd, Bva, Lva, enc_base, train=False)

### 4.3 Improved SSD: adding / removing layers

| Change | Why |
|---|---|
| **+ BatchNorm + a second conv** in every backbone stage (2 convs per stage) | deeper, easier-to-train features |
| **+ third pooling stage** (backbone output 32×32 instead of 64×64) | removes the huge 64×64 map (24,576 tiny anchors, mostly background) |
| **+ a 4th detection scale** (32, 16, 8, 4) | better coverage of large faces |
| **Downsampling convs with 256 filters + BN** | stronger extra feature layers |

In [ ]:
def build_ssd_improved(input_shape=(SSD_SIZE, SSD_SIZE, 3), num_classes=NUM_CLASSES + 1, num_boxes=SSD_BOXES):
    inputs = tf.keras.Input(shape=input_shape)
    x = inputs
    for f in (32, 64, 128):                              # 3 stages, 2 convs each
        x = conv_bn(x, f); x = conv_bn(x, f); x = layers.MaxPooling2D(2)(x)
    class_outputs, box_outputs, fmap_sizes = [], [], []
    for i in range(4):                                   # 4 detection scales: 32, 16, 8, 4
        fmap_sizes.append(x.shape[1])
        c, b = ssd_head(x, num_classes, num_boxes)
        class_outputs.append(c); box_outputs.append(b)
        if i < 3:
            x = conv_bn(x, 256, 3, 2)
    out = [layers.Concatenate(axis=1)(class_outputs), layers.Concatenate(axis=1)(box_outputs)]
    return models.Model(inputs, out, name="SSD_Improved"), fmap_sizes

ssd_imp, imp_fmaps = build_ssd_improved()
SSD_IMP_SCALES = [0.08, 0.20, 0.40, 0.65, 0.90]
anchors_imp = make_anchors(imp_fmaps, SSD_IMP_SCALES)
assert anchors_imp.shape[0] == ssd_imp.output_shape[0][1]
print("Feature maps:", imp_fmaps, " anchors:", anchors_imp.shape[0])
print("Baseline params:", f"{ssd_base.count_params():,}", " Improved params:", f"{ssd_imp.count_params():,}")

enc_imp = make_ssd_encoder(anchors_imp)
ssd_train_ds_imp = build_ds(X_tr_ssd, Btr, Ltr, enc_imp, train=True)
ssd_val_ds_imp   = build_ds(X_va_ssd, Bva, Lva, enc_imp, train=False)

In [ ]:
# Train both SSD versions from scratch
ssd_cfg = {
    'SSD baseline (tutorial)': (ssd_base, anchors_base, ssd_train_ds_base, ssd_val_ds_base, 5e-4),
    'SSD improved':            (ssd_imp,  anchors_imp,  ssd_train_ds_imp,  ssd_val_ds_imp,  1e-3),
}
for name, (m, anc, tr, va, lr) in ssd_cfg.items():
    tf.random.set_seed(SEED)
    hist, t = train_model(m, ssd_loss, tr, va, EPOCHS_SCRATCH, lr, name)
    preds = predict_detections(m, X_va_ssd, partial(decode_ssd, anchors=anc))
    register(name, preds, m.count_params(), t, hist)
plot_history(list(ssd_cfg))

## Part 5: Pretrained YOLO - YOLOv8n (Ultralytics, COCO weights)

We convert our train/val split to YOLO text format (`class cx cy w h`, normalized), then **fine-tune** the COCO-pretrained `yolov8n.pt` on it.

In [ ]:
YOLO_DIR = '/content/yolo_data'
if os.path.exists(YOLO_DIR):
    shutil.rmtree(YOLO_DIR)

def export_yolo_format(items, split):
    img_dir, lbl_dir = f'{YOLO_DIR}/images/{split}', f'{YOLO_DIR}/labels/{split}'
    os.makedirs(img_dir); os.makedirs(lbl_dir)
    new_paths = []
    for it in items:
        base = os.path.basename(it['path']); stem = os.path.splitext(base)[0]
        shutil.copy(it['path'], f'{img_dir}/{base}'); new_paths.append(f'{img_dir}/{base}')
        with open(f'{lbl_dir}/{stem}.txt', 'w') as f:
            for b, l in zip(it['boxes'], it['labels']):
                f.write(f"{l} {(b[0]+b[2])/2:.6f} {(b[1]+b[3])/2:.6f} {b[2]-b[0]:.6f} {b[3]-b[1]:.6f}\n")
    return new_paths

export_yolo_format(train_items, 'train')
val_img_paths = export_yolo_format(val_items, 'val')

yaml_path = f'{YOLO_DIR}/data.yaml'
with open(yaml_path, 'w') as f:
    f.write(f"path: {YOLO_DIR}\ntrain: images/train\nval: images/val\nnames:\n")
    for i, n in enumerate(CLASS_NAMES):
        f.write(f"  {i}: {n}\n")
print(open(yaml_path).read())

In [ ]:
from ultralytics import YOLO

t0 = time.time()
yolo_pre = YOLO('yolov8n.pt')                     # COCO-pretrained nano model
yolo_pre.train(data=yaml_path, epochs=EPOCHS_PRETRAINED, imgsz=YOLO_SIZE, batch=BATCH, seed=SEED,
               project='/content/runs', name='yolov8n_facemask', exist_ok=True, verbose=False, plots=True)
yolo_time = time.time() - t0

best_yolo = YOLO('/content/runs/yolov8n_facemask/weights/best.pt')
ul_metrics = best_yolo.val(data=yaml_path, imgsz=YOLO_SIZE, verbose=False)
print(f"Ultralytics' own validation mAP50 = {ul_metrics.box.map50:.4f}")

# Same evaluation as every other model (our compute_map on the same validation images)
yolo_preds = []
for i in range(0, len(val_img_paths), 32):
    for r in best_yolo.predict(val_img_paths[i:i + 32], imgsz=YOLO_SIZE, conf=0.05, verbose=False):
        yolo_preds.append((r.boxes.xyxyn.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int)))
register('YOLOv8n pretrained', yolo_preds, sum(p.numel() for p in best_yolo.model.parameters()), yolo_time)

## Part 6: Pretrained SSD - SSDLite320 MobileNetV3-Large (torchvision, COCO weights)

We load the COCO-pretrained detector, **replace its classification head** for our 3 classes + background, and fine-tune the whole network with SGD.

In [ ]:
import torch, torch.nn as nn, torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import ssdlite320_mobilenet_v3_large, SSDLite320_MobileNet_V3_Large_Weights
from torchvision.models.detection.ssdlite import SSDLiteClassificationHead
from torchvision.models.detection import _utils as det_utils

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("torch device:", device)

def build_pretrained_ssd(num_classes_with_bg):
    m = ssdlite320_mobilenet_v3_large(weights=SSDLite320_MobileNet_V3_Large_Weights.COCO_V1)
    in_channels = det_utils.retrieve_out_channels(m.backbone, (320, 320))
    num_anchors = m.anchor_generator.num_anchors_per_location()
    norm_layer = partial(nn.BatchNorm2d, eps=0.001, momentum=0.03)
    m.head.classification_head = SSDLiteClassificationHead(in_channels, num_anchors, num_classes_with_bg, norm_layer)
    return m

class DetDS(Dataset):
    def __init__(self, items, train):
        self.items, self.train = items, train
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        it = self.items[i]
        img = cv2.cvtColor(cv2.imread(it['path']), cv2.COLOR_BGR2RGB); H, W = img.shape[:2]
        boxes = np.array(it['boxes'], np.float32) * np.array([W, H, W, H], np.float32)
        labels = np.array(it['labels'], np.int64) + 1                  # 0 is background in torchvision
        if self.train and random.random() < 0.5:
            img = img[:, ::-1].copy(); boxes[:, [0, 2]] = W - boxes[:, [2, 0]]
        t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        return t, {'boxes': torch.from_numpy(boxes), 'labels': torch.from_numpy(labels)}

collate = lambda b: tuple(zip(*b))

@torch.no_grad()
def predict_torch_ssd(model, items):
    model.eval(); out = []
    for i in range(0, len(items), 16):
        imgs, _ = collate([DetDS(items, False)[j] for j in range(i, min(i + 16, len(items)))])
        res = model([im.to(device) for im in imgs])
        for im, r in zip(imgs, res):
            H, W = im.shape[1:]
            out.append((r['boxes'].cpu().numpy() / np.array([W, H, W, H], np.float32),
                        r['scores'].cpu().numpy(), r['labels'].cpu().numpy() - 1))
    return out

ssd_pre = build_pretrained_ssd(NUM_CLASSES + 1).to(device)
print("Trainable params:", f"{sum(p.numel() for p in ssd_pre.parameters() if p.requires_grad):,}")

In [ ]:
loader = DataLoader(DetDS(train_items, True), batch_size=BATCH, shuffle=True, collate_fn=collate, num_workers=2)
params = [p for p in ssd_pre.parameters() if p.requires_grad]
opt = torch.optim.SGD(params, lr=0.01, momentum=0.9, weight_decay=4e-5)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_PRETRAINED)

ssd_hist = {'loss': [], 'val_map': []}
t0 = time.time()
for ep in range(EPOCHS_PRETRAINED):
    ssd_pre.train(); tot = 0.0
    for imgs, tgts in loader:
        imgs = [i.to(device) for i in imgs]
        tgts = [{k: v.to(device) for k, v in t.items()} for t in tgts]
        loss = sum(ssd_pre(imgs, tgts).values())
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 10.0); opt.step()
        tot += loss.item()
    sched.step()
    ssd_hist['loss'].append(tot / len(loader))
    if ep % 5 == 4 or ep == EPOCHS_PRETRAINED - 1:
        mp, _ = compute_map(predict_torch_ssd(ssd_pre, val_items), val_items)
        ssd_hist['val_map'].append(mp)
        print(f"[SSDLite] epoch {ep+1:3d}/{EPOCHS_PRETRAINED}  train_loss={ssd_hist['loss'][-1]:.4f}  val mAP@0.5={mp:.4f}")
ssd_time = time.time() - t0

register('SSDLite320 pretrained', predict_torch_ssd(ssd_pre, val_items),
         sum(p.numel() for p in ssd_pre.parameters()), ssd_time)

## Part 7: Comparison and visual results

All six models were trained on the same 80 % split and evaluated on the same 20 % validation images with the same mAP@0.5 function.

In [ ]:
df = pd.DataFrame(RESULTS).T
df = df[['mAP@0.5'] + [f'AP {c}' for c in CLASS_NAMES] + ['params', 'train_time_s']]
df['params'] = df['params'].astype(int)
display(df.sort_values('mAP@0.5', ascending=False).round(4))

ax = df['mAP@0.5'].sort_values().plot(kind='barh', figsize=(8, 4), color='steelblue')
ax.set_xlabel('mAP@0.5 (validation)'); ax.set_title('Model comparison'); plt.tight_layout(); plt.show()

In [ ]:
# Side-by-side predictions on validation images
SHOW_THR = 0.35                      # confidence threshold used only for drawing
SHOW_IDX = [0, 1, 2, 3]
model_names = list(ALL_PREDS.keys())

fig, ax = plt.subplots(len(SHOW_IDX), 1 + len(model_names), figsize=(3.2 * (1 + len(model_names)), 3.2 * len(SHOW_IDX)))
for r, idx in enumerate(SHOW_IDX):
    it = val_items[idx]
    im = cv2.resize(cv2.cvtColor(cv2.imread(it['path']), cv2.COLOR_BGR2RGB), (400, 400))
    ax[r, 0].imshow(draw_boxes(im, it['boxes'], it['labels'])); ax[r, 0].axis('off')
    if r == 0: ax[r, 0].set_title("Ground truth", fontsize=9)
    for c, n in enumerate(model_names):
        b, s, k = ALL_PREDS[n][idx]
        ax[r, c + 1].imshow(draw_boxes(im, b, k, s, SHOW_THR)); ax[r, c + 1].axis('off')
        if r == 0: ax[r, c + 1].set_title(n, fontsize=9)
plt.suptitle(f"Detections with confidence >= {SHOW_THR}   (green = with_mask, red = without_mask, yellow = incorrect)")
plt.tight_layout(); plt.show()

## Discussion (write your answers here for the lab report)

1. **Tutorial vs improved (from scratch):** which changes helped YOLO and SSD the most (BatchNorm, extra convs, removing the 64×64 map, 4th scale)? Compare mAP and parameter counts from the table.
2. **From scratch vs pretrained:** how large is the gap, and why? (pretrained backbones already know edges, textures and faces from COCO; only 682 training images are available).
3. **YOLO vs SSD:** compare accuracy, model size and training time. Which class is hardest (`mask_weared_incorrect` has far fewer examples) and why?
4. **Failure cases:** look at the image grid - small faces, crowds, occlusion. What would you change (more data, augmentation, larger input size, class weighting)?
5. **Ideas to try next:** more scratch epochs, mosaic/colour augmentation, `yolov8s`, `ssd300_vgg16`, a different Kaggle dataset.